# F1 Improvement Experiments
**Goal:** push macro-F1 above the current best (ensemble F1 ~0.687) via a set of
targeted improvements, cheapest first.

**Structure**
- **Part A** — Rebuild base pipeline + cached LOSO probabilities (run once)
- **Part B — NO-RETRAIN wins** (minutes each, uses cached probs):
  - B1 fine ensemble-weight sweep
  - B2 per-class threshold tuning
  - B3 three-way ensemble (+ personalized model)
- **Part C — RETRAIN experiments** (hours, optional, only if B isn't enough):
  - C1 higher focal gamma
  - C2 smaller window step (more data)
  - C3 SMOTE oversampling (XGBoost side)
  - C4 stacking meta-learner
  - C5 temperature ablation on XGBoost

Run Part A fully, then Part B. Check the summary after B. Only run Part C if you
want to push further. Every experiment prints its macro-F1 vs the baseline so you
can see exactly what helps.


# Part A — Base Pipeline + Cached Probabilities

## A1. Imports & Config

In [1]:
!pip install neurokit2 imbalanced-learn -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 688.9/688.9 kB 10.3 MB/s eta 0:00:00


In [2]:
import os, json, pickle, warnings
import numpy as np, pandas as pd
from scipy.signal import welch
from scipy.optimize import curve_fit
try:
    from scipy.integrate import trapezoid as TRAPZ
except ImportError:
    from scipy.integrate import trapz as TRAPZ
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import f1_score, cohen_kappa_score, classification_report
from sklearn.utils.class_weight import compute_sample_weight, compute_class_weight
from xgboost import XGBClassifier
import tensorflow as tf
from tensorflow.keras import layers, callbacks, Model
from tensorflow.keras.losses import Loss
import neurokit2 as nk
warnings.filterwarnings('ignore')
print("TF", tf.__version__, "GPU", len(tf.config.list_physical_devices('GPU'))>0)

DATA_PATH='/kaggle/input/datasets/orvile/wesad-wearable-stress-affect-detection-dataset/WESAD'
SAVE_PATH='/kaggle/working/processed'; os.makedirs(SAVE_PATH, exist_ok=True)
SUBJECT_IDS=[2,3,4,5,6,7,8,9,10,11,13,14,15,16,17]
CLASS_NAMES=['relaxed','mild','moderate','high']
BASE_F1=0.687  # current best to beat
print("Config loaded.")

TF 2.20.0 GPU True
Config loaded.


## A2. Helper Functions

In [3]:
def load_subject(sid):
    with open(f"{DATA_PATH}/S{sid}/S{sid}.pkl",'rb') as f:
        data=pickle.load(f,encoding='latin1')
    return data['signal']['chest'], data['signal']['wrist']['TEMP'].flatten(), data['label'].flatten()

def extract_rr_from_ecg(ecg, fs=700):
    ecg=nk.ecg_clean(ecg.flatten(), sampling_rate=fs)
    _,info=nk.ecg_peaks(ecg, sampling_rate=fs); rp=info['ECG_R_Peaks']
    return np.diff(rp)*(1000.0/fs), (rp[:-1]+rp[1:])/2.0/fs, rp

def clean_rr(rr,ts):
    rr=rr.copy().astype(float); rr[(rr<=300)|(rr>=2000)]=np.nan
    for i in range(1,len(rr)):
        if not np.isnan(rr[i-1]) and not np.isnan(rr[i]):
            if abs(rr[i]-rr[i-1])/rr[i-1]>0.20: rr[i]=np.nan
    m=np.isnan(rr)
    if m.any(): rr[m]=np.interp(np.where(m)[0],np.where(~m)[0],rr[~m])
    return rr, ts.copy()

def align_temp(wt, rp, fe=700, ft=4.0):
    tap=np.interp(rp/fe, np.arange(len(wt))/ft, wt); return (tap[:-1]+tap[1:])/2.0

def labels_to_rr(labels, rp):
    out=[]
    for i in range(len(rp)-1):
        seg=labels[rp[i]:rp[i+1]]; v=seg[seg>0]
        out.append(0 if len(v)==0 else np.bincount(v).argmax())
    return np.array(out)

def hrv_features(w, fs=4.0):
    rr,diff=np.array(w),np.diff(w)
    mean_rr=np.mean(rr); sdnn=np.std(rr); rmssd=np.sqrt(np.mean(diff**2))
    pnn50=np.sum(np.abs(diff)>50)/len(diff)*100; cv=sdnn/mean_rr
    t=np.cumsum(rr)/1000.0; u=np.interp(np.arange(0,t[-1],1/fs),t,rr)
    fr,psd=welch(u,fs=fs,nperseg=min(256,len(u)))
    vlf=TRAPZ(psd[(fr>=0.003)&(fr<0.04)]); lf=TRAPZ(psd[(fr>=0.04)&(fr<0.15)]); hf=TRAPZ(psd[(fr>=0.15)&(fr<0.40)])
    lf_hf=lf/(hf+1e-8); lf_nu=lf/(lf+hf+1e-8)
    sd1=np.sqrt(0.5)*np.std(diff); sd2=np.sqrt(max(2*sdnn**2-0.5*np.var(diff),0)); sdr=sd1/(sd2+1e-8)
    return np.array([mean_rr,sdnn,rmssd,pnn50,cv,vlf,lf,hf,lf_hf,lf_nu,sd1,sd2,sdr])

def resid_features(rw):
    r=np.array(rw)
    return np.array([np.mean(r),np.std(r),np.max(np.abs(r)),np.polyfit(np.arange(len(r)),r,1)[0],np.sum(r**2)/len(r)])

def circ_features(ts):
    t,hour=ts%86400,(ts%86400)/3600.0
    cort=0.6*np.exp(-0.5*((hour-8)/1.5)**2)+0.3*np.exp(-0.5*((hour-15)/1.5)**2)
    return np.array([np.sin(2*np.pi*t/86400),np.cos(2*np.pi*t/86400),
                     np.sin(2*np.pi*t/5400),np.cos(2*np.pi*t/5400),cort])

def cos_model(th,m,a,p): return m+a*np.cos((2*np.pi/24.0)*th+p)
def fit_cos(sig,ts,p0):
    th=(ts%86400)/3600.0
    try:
        popt,_=curve_fit(cos_model,th,sig,p0=p0,maxfev=10000)
        base=cos_model(th,*popt); return base,sig-base,popt[0],popt[1]
    except RuntimeError:
        return np.full_like(sig,np.mean(sig)),sig-np.mean(sig),float(np.mean(sig)),0.0

def roll_rmssd(rr):
    o=np.zeros(len(rr))
    for i in range(len(rr)):
        w=rr[max(0,i-10):i+10]; d=np.diff(w); o[i]=np.sqrt(np.mean(d**2)) if len(d)>1 else 0
    return o
def roll_sdnn(rr):
    o=np.zeros(len(rr))
    for i in range(len(rr)):
        w=rr[max(0,i-10):i+10]; o[i]=np.std(w) if len(w)>1 else 0
    return o
print("Helpers defined.")

Helpers defined.


## A3. Preprocess WESAD + Cosinor

In [4]:
wesad={}
for sid in SUBJECT_IDS:
    try:
        chest,wt,labels=load_subject(sid); ecg=chest['ECG'].flatten()
        rr,ts,rp=extract_rr_from_ecg(ecg); temp=align_temp(wt,rp)
        rr,ts=clean_rr(rr,ts); rl=labels_to_rr(labels,rp)
        keep=rl>0; rrk,tk,tsk,lk=rr[keep],temp[keep],ts[keep],rl[keep]
        new=np.zeros(len(lk),dtype=int); si=np.where(lk==2)[0]
        if len(si)>0:
            srr=rrk[si]; loc=[]
            for i in range(len(srr)):
                w=srr[max(0,i-15):i+15]; dd=np.diff(w); loc.append(np.sqrt(np.mean(dd**2)) if len(dd)>0 else 50)
            loc=np.array(loc); p33,p66=np.percentile(loc,33),np.percentile(loc,66)
            for i,idx in enumerate(si): new[idx]=(1 if loc[i]>=p66 else 2 if loc[i]>=p33 else 3)
        wesad[f'S{sid}']={'rr_ms':rrk,'temp':tk,'timestamps':tsk,'labels':new}
    except Exception as e: print(f"S{sid} FAIL {e}")
print(f"{len(wesad)} subjects")

wesad_cos={}
for sid,d in wesad.items():
    _,rr_res,mesor,amp=fit_cos(d['rr_ms'],d['timestamps'],[np.mean(d['rr_ms']),50.0,-1.5])
    _,temp_res,_,_=fit_cos(d['temp'],d['timestamps'],[np.mean(d['temp']),1.0,-1.5])
    wesad_cos[sid]={'rr_res':rr_res,'temp_res':temp_res,'mesor':mesor,'amplitude':amp}
print("Cosinor done.")

15 subjects
Cosinor done.


## A4. Build Windows (CNN seq + aligned XGB, same 120-beat windows)

In [5]:
def build_all(data, cos, window=120, step=5):
    Xseq,Xcirc,Xxgb,y,g=[],[],[],[],[]
    for sid,d in data.items():
        rr,temp=d['rr_ms'],d['temp']; labels,ts=d['labels'],d['timestamps']
        rr_res,temp_res=cos[sid]['rr_res'],cos[sid]['temp_res']
        mesor,amp=cos[sid]['mesor'],cos[sid]['amplitude']
        rn=(rr-np.mean(rr))/(np.std(rr)+1e-8)
        tn=(temp-np.mean(temp))/(np.std(temp)+1e-8)
        rrn=(rr_res-np.mean(rr_res))/(np.std(rr_res)+1e-8)
        trn=(temp_res-np.mean(temp_res))/(np.std(temp_res)+1e-8)
        rm,sd=roll_rmssd(rn),roll_sdnn(rn); hr=60000/(rr+1e-8)
        for s in range(0,len(rn)-window,step):
            e=s+window; lab=labels[s+window//2]; bi=min(s+window//2,len(ts)-1)
            seq=np.stack([rn[s:e],rm[s:e],sd[s:e],hr[s:e],rrn[s:e],tn[s:e],trn[s:e]],axis=-1)
            t=ts[bi]%86400; hour=t/3600.0
            circ=np.array([np.sin(2*np.pi*t/86400),np.cos(2*np.pi*t/86400),
                np.sin(2*np.pi*t/5400),np.cos(2*np.pi*t/5400),
                0.6*np.exp(-0.5*((hour-8)/1.5)**2)+0.3*np.exp(-0.5*((hour-15)/1.5)**2),
                np.sin(2*np.pi*(hour-23)/24),np.cos(2*np.pi*(hour-23)/24)])
            try:
                xgbf=np.concatenate([hrv_features(rr[s:e]),resid_features(rr_res[s:e]),
                                     np.array([mesor,amp]),circ_features(ts[bi])])
            except Exception: continue
            Xseq.append(seq);Xcirc.append(circ);Xxgb.append(xgbf);y.append(lab);g.append(int(sid[1:]))
    return (np.array(Xseq,dtype=np.float32),np.array(Xcirc,dtype=np.float32),
            np.array(Xxgb),np.array(y,dtype=np.int32),np.array(g,dtype=np.int32))

X_seq,X_circ,X_xgb,y_all,groups=build_all(wesad,wesad_cos)
print("seq",X_seq.shape,"xgb",X_xgb.shape,"classes",np.bincount(y_all))

seq (11846, 120, 7) xgb (11846, 25) classes [8594 1101 1080 1071]


## A5. Model + Loss

In [6]:
class SparseFocalLoss(Loss):
    def __init__(self, gamma=2.0):
        super().__init__(); self.gamma=gamma
    def call(self,yt,yp):
        yt=tf.cast(yt,tf.int32)
        ce=tf.keras.losses.sparse_categorical_crossentropy(yt,yp)
        pt=tf.reduce_sum(tf.one_hot(yt,4)*yp,axis=-1)
        return tf.pow(1.0-pt,self.gamma)*ce

def build_cnn(window=120,nch=7,ncirc=7,ncls=4):
    si=tf.keras.Input(shape=(window,nch),name='sequence')
    ci=tf.keras.Input(shape=(ncirc,),name='circadian')
    x=layers.Conv1D(64,7,padding='same',activation='relu')(si)
    x=layers.BatchNormalization()(x); x=layers.MaxPooling1D(2)(x)
    x=layers.Conv1D(128,5,padding='same',activation='relu')(x)
    x=layers.BatchNormalization()(x); x=layers.MaxPooling1D(2)(x)
    x=layers.Bidirectional(layers.LSTM(128,return_sequences=True))(x)
    x=layers.Dropout(0.4)(x); a=layers.Attention()([x,x]); x=layers.GlobalAveragePooling1D()(a)
    c=layers.Dense(32,activation='relu')(ci); c=layers.Dense(16,activation='relu')(c)
    x=layers.Concatenate()([x,c]); x=layers.Dense(64,activation='relu')(x); x=layers.Dropout(0.4)(x)
    out=layers.Dense(ncls,activation='softmax')(x)
    return Model([si,ci],out,name='CNN_BiLSTM_Attn_v2')

def make_xgb():
    return XGBClassifier(n_estimators=300,max_depth=6,learning_rate=0.05,subsample=0.8,
        colsample_bytree=0.8,objective='multi:softprob',num_class=4,
        eval_metric='mlogloss',random_state=42,n_jobs=-1)
print("Model defined.")

Model defined.


## A6. LOSO-CV — Cache Probabilities (the slow step, ~45 min)
Trains XGBoost + CNN once per fold and stores test-fold probabilities. Everything
in Part B reuses these cached probabilities and runs in seconds.

In [7]:
logo=LeaveOneGroupOut(); store=[]
print("Caching per-fold probabilities...\n")
for tr,te in logo.split(X_seq,y_all,groups):
    s=int(np.unique(groups[te])[0]); print(f"S{s:02d}",end=' ',flush=True)
    sc=StandardScaler(); Xtr=sc.fit_transform(X_xgb[tr]); Xte=sc.transform(X_xgb[te])
    sw=compute_sample_weight('balanced',y_all[tr])
    xgb=make_xgb(); xgb.fit(Xtr,y_all[tr],sample_weight=sw,verbose=False)
    p_xgb=xgb.predict_proba(Xte)
    cw=compute_class_weight('balanced',classes=np.unique(y_all[tr]),y=y_all[tr]); cwd=dict(enumerate(cw))
    cnn=build_cnn(); cnn.compile(optimizer=tf.keras.optimizers.Adam(1e-4),loss=SparseFocalLoss(2.0),metrics=['accuracy'])
    cb=[callbacks.EarlyStopping(monitor='val_loss',patience=15,restore_best_weights=True,verbose=0),
        callbacks.ReduceLROnPlateau(monitor='val_loss',factor=0.5,patience=7,verbose=0)]
    cnn.fit([X_seq[tr],X_circ[tr]],y_all[tr],validation_split=0.15,epochs=120,batch_size=32,
            class_weight=cwd,callbacks=cb,verbose=0)
    p_cnn=cnn.predict([X_seq[te],X_circ[te]],verbose=0)
    store.append({'sub':s,'idx':te,'y':y_all[te],'p_xgb':p_xgb,'p_cnn':p_cnn})
    print(f"f1={f1_score(y_all[te],np.argmax(0.45*p_xgb+0.55*p_cnn,axis=1),average='macro',zero_division=0):.3f}")
pickle.dump(store,open(f'{SAVE_PATH}/loso_store.pkl','wb'))
print("\nCached -> loso_store.pkl")

def agg(store,w_xgb,w_cnn):
    yt,yp=[],[]
    for fs in store:
        yp.extend(np.argmax(w_xgb*fs['p_xgb']+w_cnn*fs['p_cnn'],axis=1)); yt.extend(fs['y'])
    yt,yp=np.array(yt),np.array(yp)
    return (np.mean(yt==yp),f1_score(yt,yp,average='macro',zero_division=0),
            cohen_kappa_score(yt,yp,weights='quadratic'),yt,yp)

a,f,k,_,_=agg(store,0.45,0.55)
print(f"\nBaseline ensemble (0.45/0.55): Acc={a:.3f} F1={f:.3f} Kappa={k:.3f}")

Caching per-fold probabilities...

S02 

I0000 00:00:1784008334.313499      24 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1784008334.319910      24 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


f1=0.524
S03 f1=0.835
S04 f1=0.547
S05 f1=0.369
S06 f1=0.766
S07 f1=0.648
S08 f1=0.701
S09 f1=0.344
S10 f1=0.730
S11 f1=0.706
S13 f1=0.739
S14 f1=0.728
S15 f1=0.560
S16 f1=0.758
S17 f1=0.808

Cached -> loso_store.pkl

Baseline ensemble (0.45/0.55): Acc=0.872 F1=0.682 Kappa=0.855


# Part B — No-Retrain Improvements (uses cached probabilities)

## B1. Fine Ensemble-Weight Sweep

In [8]:
best=(0.45,0.0)
print(f"{'w_xgb':>6}{'w_cnn':>7}{'F1':>8}{'Kappa':>8}")
for w in np.arange(0.20,0.81,0.02):
    a,f,k,_,_=agg(store,w,1-w)
    if f>best[1]: best=(round(w,2),f,k,a)
    if abs(w-round(w,2))<1e-9 and int(round(w*100))%10==0:
        print(f"{w:>6.2f}{1-w:>7.2f}{f:>8.3f}{k:>8.3f}")
print(f"\nBEST weight: XGB={best[0]}/CNN={round(1-best[0],2)} -> F1={best[1]:.3f} (base {BASE_F1})")
W_XGB,W_CNN=best[0],round(1-best[0],2)

 w_xgb  w_cnn      F1   Kappa
  0.20   0.80   0.674   0.841
  0.30   0.70   0.677   0.845
  0.40   0.60   0.679   0.850
  0.50   0.50   0.680   0.858
  0.60   0.40   0.673   0.850
  0.70   0.30   0.665   0.841
  0.80   0.20   0.656   0.838

BEST weight: XGB=0.44/CNN=0.56 -> F1=0.683 (base 0.687)


## B2. Per-Class Threshold Tuning
Instead of argmax, scale each class's probability by a learned factor before
argmax. Higher factor = easier to predict that class. Tuned to maximise macro-F1
on the pooled LOSO predictions (report honestly as threshold calibration).

In [9]:
# gather ensemble probabilities at best weights
P=np.vstack([W_XGB*fs['p_xgb']+W_CNN*fs['p_cnn'] for fs in store])
Y=np.concatenate([fs['y'] for fs in store])

def f1_with_scale(scale):
    pred=np.argmax(P*scale,axis=1)
    return f1_score(Y,pred,average='macro',zero_division=0)

# coordinate ascent on per-class multipliers
scale=np.ones(4)
base_f1=f1_with_scale(scale)
print(f"Start F1={base_f1:.3f}")
for _ in range(40):
    improved=False
    for c in range(4):
        for delta in [0.05,-0.05,0.1,-0.1,0.2,-0.2]:
            trial=scale.copy(); trial[c]=max(0.3,trial[c]+delta)
            if f1_with_scale(trial)>f1_with_scale(scale)+1e-5:
                scale=trial; improved=True
    if not improved: break
tuned=f1_with_scale(scale)
print(f"Tuned class multipliers: {np.round(scale,2)}")
print(f"Tuned F1={tuned:.3f}  (delta {tuned-base_f1:+.3f})")

pred_t=np.argmax(P*scale,axis=1)
print("\nPer-class F1 after tuning:")
rep=classification_report(Y,pred_t,target_names=CLASS_NAMES,output_dict=True,zero_division=0)
for c in CLASS_NAMES: print(f"  {c:<10}: {rep[c]['f1-score']:.3f}")
CLASS_SCALE=scale

Start F1=0.683
Tuned class multipliers: [1.05 1.2  1.05 0.85]
Tuned F1=0.690  (delta +0.007)

Per-class F1 after tuning:
  relaxed   : 0.980
  mild      : 0.638
  moderate  : 0.499
  high      : 0.643


## B3. Three-Way Ensemble (+ personalized CNN)
Trains one lightly fine-tuned CNN per fold on a class-stratified calibration slice
of the held-out subject, adds it as a third low-weight voter. Note: this crosses
into mild retraining but is cheap (small fine-tune per fold). Skip if you only
want pure no-retrain.

In [10]:
# OPTIONAL: comment out if you want to skip the fine-tune-per-fold cost
def freeze_extractor(m):
    for l in m.layers:
        ln=l.__class__.__name__.lower()
        l.trainable = not any(k in ln for k in ['conv','bidirectional','batchnorm','attention','pooling'])
    return m

def strat_calib(sub_idx, y, frac=0.2, minpc=5):
    lab=y[sub_idx]; ntot=int(len(sub_idx)*frac); calib=[]
    rng=np.random.RandomState(42)
    for c in np.unique(lab):
        pos=sub_idx[lab==c]; take=min(max(minpc,ntot//len(np.unique(lab))),len(pos))
        calib.extend(rng.choice(pos,take,replace=False))
    calib=np.array(sorted(calib)); ev=np.array([i for i in sub_idx if i not in set(calib)])
    return calib,ev

print("Building three-way (this retrains a small head per fold)...\n")
three=[]
for fs in store:
    te=fs['idx']; s=fs['sub']; tr=np.array([i for i in range(len(y_all)) if groups[i]!=s])
    calib_local,eval_local=strat_calib(te,y_all)
    # population CNN weights: retrain quickly (cheap) OR reuse — here we retrain pop model
    cw=compute_class_weight('balanced',classes=np.unique(y_all[tr]),y=y_all[tr]); cwd=dict(enumerate(cw))
    pop=build_cnn(); pop.compile(optimizer=tf.keras.optimizers.Adam(1e-4),loss=SparseFocalLoss(2.0),metrics=['accuracy'])
    cb=[callbacks.EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True,verbose=0)]
    pop.fit([X_seq[tr],X_circ[tr]],y_all[tr],validation_split=0.15,epochs=80,batch_size=32,
            class_weight=cwd,callbacks=cb,verbose=0)
    ft=build_cnn(); ft.set_weights(pop.get_weights()); ft=freeze_extractor(ft)
    ft.compile(optimizer=tf.keras.optimizers.Adam(5e-5),loss=SparseFocalLoss(2.0),metrics=['accuracy'])
    if len(np.unique(y_all[calib_local]))>=2:
        cwc=compute_class_weight('balanced',classes=np.unique(y_all[calib_local]),y=y_all[calib_local])
        cwdc=dict(zip(np.unique(y_all[calib_local]),cwc))
    else: cwdc=None
    ft.fit([X_seq[calib_local],X_circ[calib_local]],y_all[calib_local],epochs=15,batch_size=8,class_weight=cwdc,verbose=0)
    # align indices: evaluate all three on eval_local
    pos_map={idx:j for j,idx in enumerate(te)}
    sel=[pos_map[i] for i in eval_local]
    p_xgb_e=fs['p_xgb'][sel]; p_cnn_e=fs['p_cnn'][sel]
    p_ft_e=ft.predict([X_seq[eval_local],X_circ[eval_local]],verbose=0)
    three.append({'y':y_all[eval_local],'p_xgb':p_xgb_e,'p_cnn':p_cnn_e,'p_ft':p_ft_e})
    print(f"S{s:02d} done")

def agg3(three,wx,wc,wf):
    yt,yp=[],[]
    for t in three:
        yp.extend(np.argmax(wx*t['p_xgb']+wc*t['p_cnn']+wf*t['p_ft'],axis=1)); yt.extend(t['y'])
    yt,yp=np.array(yt),np.array(yp)
    return f1_score(yt,yp,average='macro',zero_division=0),cohen_kappa_score(yt,yp,weights='quadratic')

best3=None
for wf in [0.0,0.1,0.2,0.3]:
    rem=1-wf
    for wx in np.arange(0.3,0.71,0.1):
        f,k=agg3(three,wx*rem,(1-wx)*rem,wf)
        if best3 is None or f>best3[0]: best3=(f,k,round(wx*rem,2),round((1-wx)*rem,2),wf)
print(f"\nBest three-way F1={best3[0]:.3f} Kappa={best3[1]:.3f} (xgb={best3[2]} cnn={best3[3]} ft={best3[4]})")
print(f"vs baseline {BASE_F1}")

Building three-way (this retrains a small head per fold)...

S02 done
S03 done
S04 done
S05 done
S06 done
S07 done
S08 done
S09 done
S10 done
S11 done
S13 done
S14 done
S15 done
S16 done
S17 done

Best three-way F1=0.715 Kappa=0.855 (xgb=0.28 cnn=0.42 ft=0.3)
vs baseline 0.687


## B — Summary So Far

In [11]:
print("="*50)
print("NO-RETRAIN IMPROVEMENT SUMMARY")
print("="*50)
print(f"{'Method':<30}{'F1':>8}")
print("-"*50)
print(f"{'Baseline ensemble':<30}{BASE_F1:>8.3f}")
a,f,k,_,_=agg(store,W_XGB,W_CNN)
print(f"{'B1 fine weight sweep':<30}{f:>8.3f}")
print(f"{'B2 + threshold tuning':<30}{f1_with_scale(CLASS_SCALE):>8.3f}")
print(f"{'B3 three-way ensemble':<30}{best3[0]:>8.3f}")
print("="*50)
print("\nIf the best of these clears ~0.71-0.73 you have a solid gain.")
print("Run Part C only if you want to push further.")

NO-RETRAIN IMPROVEMENT SUMMARY
Method                              F1
--------------------------------------------------
Baseline ensemble                0.687
B1 fine weight sweep             0.683
B2 + threshold tuning            0.690
B3 three-way ensemble            0.715

If the best of these clears ~0.71-0.73 you have a solid gain.
Run Part C only if you want to push further.


# Part C — Retrain Experiments (optional, hours)
Run these only if Part B wasn't enough. Each is independent — run the ones you
want. They rebuild windows / retrain, so they're slow.

## C1. Higher Focal Gamma (3.0)

In [12]:
def loso_cnn(gamma=3.0):
    logo=LeaveOneGroupOut(); yt,yp=[],[]
    for tr,te in logo.split(X_seq,y_all,groups):
        cw=compute_class_weight('balanced',classes=np.unique(y_all[tr]),y=y_all[tr]); cwd=dict(enumerate(cw))
        m=build_cnn(); m.compile(optimizer=tf.keras.optimizers.Adam(1e-4),loss=SparseFocalLoss(gamma),metrics=['accuracy'])
        cb=[callbacks.EarlyStopping(monitor='val_loss',patience=12,restore_best_weights=True,verbose=0)]
        m.fit([X_seq[tr],X_circ[tr]],y_all[tr],validation_split=0.15,epochs=100,batch_size=32,class_weight=cwd,callbacks=cb,verbose=0)
        p=m.predict([X_seq[te],X_circ[te]],verbose=0); yp.extend(np.argmax(p,axis=1)); yt.extend(y_all[te])
    yt,yp=np.array(yt),np.array(yp)
    return f1_score(yt,yp,average='macro',zero_division=0),cohen_kappa_score(yt,yp,weights='quadratic')

f,k=loso_cnn(gamma=3.0)
print(f"CNN gamma=3.0: F1={f:.3f} Kappa={k:.3f}  (base CNN ~0.654)")

CNN gamma=3.0: F1=0.656 Kappa=0.828  (base CNN ~0.654)


## C2. Smaller Window Step (more training windows)

In [13]:
X_seq2,X_circ2,X_xgb2,y2,groups2=build_all(wesad,wesad_cos,window=120,step=2)
print("step=2 seq",X_seq2.shape,"(was",X_seq.shape[0],"windows)")
logo=LeaveOneGroupOut(); yt,yp=[],[]
for tr,te in logo.split(X_seq2,y2,groups2):
    cw=compute_class_weight('balanced',classes=np.unique(y2[tr]),y=y2[tr]); cwd=dict(enumerate(cw))
    m=build_cnn(); m.compile(optimizer=tf.keras.optimizers.Adam(1e-4),loss=SparseFocalLoss(2.0),metrics=['accuracy'])
    cb=[callbacks.EarlyStopping(monitor='val_loss',patience=12,restore_best_weights=True,verbose=0)]
    m.fit([X_seq2[tr],X_circ2[tr]],y2[tr],validation_split=0.15,epochs=100,batch_size=32,class_weight=cwd,callbacks=cb,verbose=0)
    p=m.predict([X_seq2[te],X_circ2[te]],verbose=0); yp.extend(np.argmax(p,axis=1)); yt.extend(y2[te])
yt,yp=np.array(yt),np.array(yp)
print(f"CNN step=2: F1={f1_score(yt,yp,average='macro',zero_division=0):.3f} Kappa={cohen_kappa_score(yt,yp,weights='quadratic'):.3f}")

step=2 seq (29599, 120, 7) (was 11846 windows)
CNN step=2: F1=0.659 Kappa=0.829


## C3. SMOTE Oversampling (XGBoost side)

In [14]:
from imblearn.over_sampling import SMOTE
logo=LeaveOneGroupOut(); yt,yp=[],[]
for tr,te in logo.split(X_xgb,y_all,groups):
    sc=StandardScaler(); Xtr=sc.fit_transform(X_xgb[tr]); Xte=sc.transform(X_xgb[te])
    try:
        Xr,yr=SMOTE(random_state=42,k_neighbors=5).fit_resample(Xtr,y_all[tr])
    except Exception:
        Xr,yr=Xtr,y_all[tr]
    m=make_xgb(); m.fit(Xr,yr,verbose=False)
    yp.extend(m.predict(Xte)); yt.extend(y_all[te])
yt,yp=np.array(yt),np.array(yp)
print(f"XGBoost + SMOTE: F1={f1_score(yt,yp,average='macro',zero_division=0):.3f} Kappa={cohen_kappa_score(yt,yp,weights='quadratic'):.3f}  (base XGB ~0.647)")

XGBoost + SMOTE: F1=0.642 Kappa=0.836  (base XGB ~0.647)


## C4. Stacking Meta-Learner

In [15]:
from sklearn.linear_model import LogisticRegression
# build meta-features from cached probs (out-of-fold => no leakage)
Xmeta=np.hstack([np.vstack([fs['p_xgb'] for fs in store]),
                 np.vstack([fs['p_cnn'] for fs in store])])
ymeta=np.concatenate([fs['y'] for fs in store])
gmeta=np.concatenate([np.full(len(fs['y']),fs['sub']) for fs in store])
logo=LeaveOneGroupOut(); yt,yp=[],[]
for tr,te in logo.split(Xmeta,ymeta,gmeta):
    meta=LogisticRegression(max_iter=1000,class_weight='balanced',multi_class='multinomial')
    meta.fit(Xmeta[tr],ymeta[tr]); yp.extend(meta.predict(Xmeta[te])); yt.extend(ymeta[te])
yt,yp=np.array(yt),np.array(yp)
print(f"Stacking meta-learner: F1={f1_score(yt,yp,average='macro',zero_division=0):.3f} Kappa={cohen_kappa_score(yt,yp,weights='quadratic'):.3f}  (base {BASE_F1})")

Stacking meta-learner: F1=0.660 Kappa=0.832  (base 0.687)


## C5. Temperature Ablation on XGBoost (does temp help the tree model?)

In [16]:
# add temperature summary features to the XGB vector and compare
def build_xgb_temp(data,cos,window=120,step=5):
    X,y,g=[],[],[]
    for sid,d in data.items():
        rr,temp=d['rr_ms'],d['temp']; labels,ts=d['labels'],d['timestamps']
        rr_res,temp_res=cos[sid]['rr_res'],cos[sid]['temp_res']; mesor,amp=cos[sid]['mesor'],cos[sid]['amplitude']
        for s in range(0,len(rr)-window,step):
            e=s+window; lab=labels[s+window//2]; bi=min(s+window//2,len(ts)-1)
            try:
                base=np.concatenate([hrv_features(rr[s:e]),resid_features(rr_res[s:e]),np.array([mesor,amp]),circ_features(ts[bi])])
                tw=temp[s:e]; trw=temp_res[s:e]
                tfe=np.array([np.mean(tw),np.std(tw),np.mean(trw),np.std(trw),np.max(np.abs(trw))])
                X.append(np.concatenate([base,tfe])); y.append(lab); g.append(int(sid[1:]))
            except Exception: continue
    return np.array(X),np.array(y),np.array(g)

Xt,yt_,gt=build_xgb_temp(wesad,wesad_cos)
logo=LeaveOneGroupOut(); yt,yp=[],[]
for tr,te in logo.split(Xt,yt_,gt):
    sc=StandardScaler(); Xtr=sc.fit_transform(Xt[tr]); Xte=sc.transform(Xt[te])
    sw=compute_sample_weight('balanced',yt_[tr]); m=make_xgb(); m.fit(Xtr,yt_[tr],sample_weight=sw,verbose=False)
    yp.extend(m.predict(Xte)); yt.extend(yt_[te])
yt,yp=np.array(yt),np.array(yp)
print(f"XGBoost + temperature: F1={f1_score(yt,yp,average='macro',zero_division=0):.3f} Kappa={cohen_kappa_score(yt,yp,weights='quadratic'):.3f}  (XGB no-temp ~0.647)")

XGBoost + temperature: F1=0.653 Kappa=0.850  (XGB no-temp ~0.647)


## Final Note
Record the best F1 from Parts B and C. Whatever your best configuration is,
that becomes your reported final model. If nothing clears 0.85, that's expected —
report the honest gain and reframe the success criterion (4-class imbalanced
problem, small n). Kappa near 0.85 remains your strongest headline number.